# Reconstrução de Cenas 3D com Neural Radiance Fields (NeRF)
### Trabalho de Conclusão de Curso — UNIVALI

Este notebook implementa um pipeline completo de reconstrução 3D a partir de imagens ou vídeos, utilizando o framework **Nerfstudio** no Google Colab.

---

## Fluxo de Trabalho

```
Seção 0  →  Configuração Global (variáveis editáveis centralizadas)
Seção 1  →  Instalação de Dependências (executar uma vez por sessão)
Seção 2  →  Configuração do Ambiente (montar Drive, criar pastas)
Seção 3  →  Aquisição de Dados (exemplo pronto, upload de imagens ou vídeo)
Seção 4  →  (Opcional) Redimensionamento de Imagens
Seção 5  →  Processamento com COLMAP (extração de poses da câmera)
Seção 6  →  Treinamento do Modelo (nerfacto ou splatfacto)
Seção 7  →  Exportação e Geração de Artefatos (vídeo, nuvem de pontos, zip)
```

> **Dica:** Execute as seções em ordem sequencial. Após a Seção 1.1, o kernel será reiniciado — isso é esperado; basta continuar a partir da Seção 1.2.

---
## Seção 0: Instalação de Dependências

Execute esta seção **uma vez no início de cada sessão** do Colab. Ela instala o Conda, cria o ambiente `ns` e instala PyTorch, COLMAP, Nerfstudio e demais bibliotecas necessárias.

### Seção 0.1 — Instalar Conda (reinicia o kernel automaticamente)
> Após o reinício do kernel, execute a partir da **Seção 0.2** — não repita esta célula.

In [ ]:
# Célula 0.1: Instala o Conda no ambiente do Google Colab
# Isso reiniciará o kernel automaticamente. Após o reinício, prossiga para a célula 1.2.
!pip -q install condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


### Seção 0.2 — Criar ambiente e instalar dependências
> Este passo pode demorar **5–10 minutos**.

### Seção 0.3 — Verificar GPU disponível

In [ ]:
# Seção 0.2: Cria o ambiente conda e instala todas as dependências
# Este passo pode demorar 5–10 minutos. Execute-o apenas uma vez por sessão.

# Cria o ambiente isolado "ns" com Python 3.10
!mamba create -y -n ns python=3.10

# PyTorch com suporte a CUDA 11.8
!mamba install -y -n ns -c pytorch -c nvidia pytorch torchvision pytorch-cuda=11.8

# COLMAP (poses de câmera), ffmpeg (vídeos) e git
!mamba install -y -n ns -c conda-forge colmap ffmpeg git openh264

# Nerfstudio e a métrica LPIPS
!conda run -n ns python -m pip install -U pip wheel setuptools
!conda run -n ns python -m pip install nerfstudio lpips

# Node.js + Localtunnel (exposição do viewer via web) e Xvfb (display virtual headless p/ COLMAP)
!apt-get -qq update && apt-get -qq install -y nodejs npm xvfb
!npm -g -s install localtunnel

print("✓ Dependências instaladas com sucesso.")

Streaming output truncated to the last 5000 lines.
Extracting   (1)  ⣾  [+] 1m:17.9s
Extracting   (1)  ⣾  [+] 1m:18.0s
Extracting   (1)  ⣾  [+] 1m:18.1s
Extracting   (1)  ⣾  [+] 1m:18.2s
Extracting   (1)  ⣾  [+] 1m:18.3s
Extracting   (1)  ⣾  [+] 1m:18.4s
Extracting   (1)  ⣾  [+] 1m:18.5s
Extracting   (1)  ⣾  [+] 1m:18.6s
Extracting   (1)  ⣾  [+] 1m:18.7s
Extracting   (1)  ⣾  [+] 1m:18.8s
Extracting   (1)  ⣾  [+] 1m:18.9s
Extracting   (1)  ⣾  [+] 1m:19.0s
Extracting   (1)  ⣾  [+] 1m:19.1s
Extracting   (1)  ⣾  [+] 1m:19.2s
Extracting   (1)  ⣾  [+] 1m:19.3s
Extracting   (1)  ⣾  [+] 1m:19.4s
Extracting   (1)  ⣾  [+] 1m:19.5s
Extracting   (1)  ⣾  [+] 1m:19.6s
Extracting   (1)  ⣾  [+] 1m:19.7s
Extracting   (1)  ⣾  [+] 1m:19.8s
Extracting   (1)  ⣾  [+] 1m:19.9s
Extracting   (1)  ⣾  [+] 1m:20.0s
Extracting   (1)  ⣾  [+] 1m:20.1s
Extracting   (1)  ⣾  [+] 1m:20.2s
Extracting   (1)  ⣾  [+] 1m:20.3s
Extracting   (1)  ⣾  [+] 1m:20.4s
Extracting   (1)  ⣾  [+] 1m:20.5s
Extracting   (1)  ⣾  [+] 1m:20.

In [ ]:
# Seção 1.3: Verifica a GPU disponível e a versão do CUDA
!nvidia-smi
!nvcc --version

Wed Mar  4 23:25:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
## Seção 1: Configuração Global

**Edite as variáveis abaixo antes de executar qualquer outra célula.** Elas controlam os caminhos, a fonte dos dados e os parâmetros de processamento usados em todo o pipeline.

In [ ]:
# ============================================================
# SEÇÃO 1: CONFIGURAÇÃO GLOBAL — edite apenas este bloco
# ============================================================

import os

# --- Diretório base no Google Drive ---
DIRETORIO_BASE = "/content/drive/MyDrive/Faculdade/TCC/desenvolvimento"

# --- Fonte dos dados de entrada ---
# Opções: 'poster', 'dozer', 'desolation', 'UPLOAD_IMAGENS', 'UPLOAD_VIDEO'
# Caso já possua um dataset salvo no drive, defina aqui o nome da pasta (ex: "meu_dataset") e certifique-se de que a estrutura de pastas seja idêntica à dos datasets de exemplo.
# Além disso, caso já possua um dataset salvo no drive, pode pular a seção 2 (upload) e ir direto para a seção 3 (pré-processamento).
FONTE_DOS_DADOS = "poster"

# --- Nome da cena (usado como nome de pasta de saída) ---
# Para datasets de exemplo ('poster', 'dozer', 'desolation') este valor será
# sobrescrito automaticamente na Seção 3. Para uploads, defina aqui.
NOME_DA_CENA = "poster"

# --- Redimensionamento de imagens (Seção 4) ---
EXECUTAR_REDIMENSIONAMENTO = True   # Ativar apenas para UPLOAD_IMAGENS
MAX_DIMENSAO = 1600                 # Maior borda em pixels (mantém proporção)

# ============================================================
# Caminhos derivados — NÃO edite abaixo desta linha
# ============================================================
DATA_DIR    = os.path.join(DIRETORIO_BASE, "data")
DATASET_DIR = os.path.join(DIRETORIO_BASE, "datasets")
OUTPUTS_DIR = os.path.join(DIRETORIO_BASE, "outputs")
EXPORTS_DIR = os.path.join(DIRETORIO_BASE, "exports")

# Prefixo para rodar comandos dentro do ambiente conda "ns"
NS_PREFIX = "conda run -n ns"

print("✓ Configuração global carregada.")
print(f"  Fonte de dados : {FONTE_DOS_DADOS}")
print(f"  Diretório base : {DIRETORIO_BASE}")

✓ Configuração global carregada.
  Fonte de dados : poster
  Diretório base : /content/drive/MyDrive/Faculdade/TCC/desenvolvimento


---
## Seção 2: Configuração do Ambiente

Monta o Google Drive e cria a estrutura de pastas definida na Seção 0. É altamente recomendado usar o Drive para que os arquivos não sejam perdidos ao final da sessão.

### Seção 2.1 — Montar o Drive e criar diretórios

### Seção 2.2 — Criar diretório de destino do dataset

In [ ]:
# Seção 2.1: Monta o Google Drive e cria a estrutura de pastas
# Os caminhos (DATA_DIR, DATASET_DIR, etc.) foram definidos na Seção 0.

from google.colab import drive

# Monta o Google Drive em /content/drive
drive.mount('/content/drive')

# Cria os diretórios de trabalho, se ainda não existirem
for path in [DATA_DIR, DATASET_DIR, OUTPUTS_DIR, EXPORTS_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"Diretório garantido: {path}")

Mounted at /content/drive
Diretório garantido: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/data
Diretório garantido: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/datasets
Diretório garantido: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/outputs
Diretório garantido: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/exports


In [ ]:
# Seção 3.2: Define e cria o diretório de destino do dataset processado
# Este passo depende de NOME_DA_CENA, definido na Seção 3.1.

CAMINHO_DATASET = os.path.join(DATASET_DIR, NOME_DA_CENA)
os.makedirs(CAMINHO_DATASET, exist_ok=True)

print(f"✓ Diretório do dataset: {CAMINHO_DATASET}")

✓ Diretório do dataset: /content/drive/MyDrive/Faculdade/TCC/desenvolvimento/datasets/poster


---
## Seção 3: Aquisição de Dados de Entrada

Escolha **uma** das fontes de dados configurando `FONTE_DOS_DADOS` na Seção 0:

| Valor | Descrição |
|---|---|
| `'poster'` / `'dozer'` / `'desolation'` | Dataset de exemplo do Nerfstudio (download automático) |
| `'UPLOAD_IMAGENS'` | Upload de imagens próprias (.jpg, .png) |
| `'UPLOAD_VIDEO'` | Upload de um arquivo de vídeo (.mp4, .mov, etc.) |

### Seção 3.1 — Adquirir dados

In [ ]:
# Seção 3.1: Adquire os dados de entrada
# Usa FONTE_DOS_DADOS e NOME_DA_CENA definidos na Seção 0.

from google.colab import files

if FONTE_DOS_DADOS in ["poster", "dozer", "desolation"]:
    # Para datasets de exemplo, o nome da cena é o próprio valor de FONTE_DOS_DADOS
    NOME_DA_CENA = FONTE_DOS_DADOS
    # Baixa e descompacta o dataset de exemplo do Nerfstudio
    !{NS_PREFIX} ns-download-data nerfstudio --capture-name={NOME_DA_CENA}
    CAMINHO_ORIGEM = f"/content/data/nerfstudio/{NOME_DA_CENA}"

elif FONTE_DOS_DADOS == "UPLOAD_IMAGENS":
    # NOME_DA_CENA vem da Seção 0; mude lá para personalizar o nome da pasta
    CAMINHO_ORIGEM = os.path.join(DATA_DIR, NOME_DA_CENA, "images_raw")
    os.makedirs(CAMINHO_ORIGEM, exist_ok=True)
    print(f"Faça o upload de suas imagens (.jpg, .png).")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        with open(os.path.join(CAMINHO_ORIGEM, filename), 'wb') as f:
            f.write(content)
    print(f"{len(uploaded)} imagens salvas em: {CAMINHO_ORIGEM}")

elif FONTE_DOS_DADOS == "UPLOAD_VIDEO":
    VIDEO_DIR = os.path.join(DATA_DIR, NOME_DA_CENA)
    os.makedirs(VIDEO_DIR, exist_ok=True)
    print("Faça o upload do seu arquivo de vídeo (.mp4, .mov, etc.)")
    uploaded = files.upload()
    video_filename = list(uploaded.keys())[0]
    CAMINHO_ORIGEM = os.path.join(VIDEO_DIR, video_filename)
    os.rename(video_filename, CAMINHO_ORIGEM)

else:
    raise ValueError(f"FONTE_DOS_DADOS inválida: '{FONTE_DOS_DADOS}'. "
                     "Opções: 'poster', 'dozer', 'desolation', 'UPLOAD_IMAGENS', 'UPLOAD_VIDEO'.")

print(f"\n✓ Fonte de dados definida.")
print(f"  Nome da cena   : {NOME_DA_CENA}")
print(f"  Caminho origem : {CAMINHO_ORIGEM}")

---
## Seção 4: (Opcional) Redimensionamento de Imagens

Se você fez upload de imagens de alta resolução (`UPLOAD_IMAGENS`), esta etapa as redimensiona para a `MAX_DIMENSAO` definida na Seção 0, reduzindo o uso de memória e o tempo de processamento sem perda significativa de qualidade.

> Ative configurando `EXECUTAR_REDIMENSIONAMENTO = True` na Seção 0.

### Seção 4.1 — Redimensionar imagens

In [ ]:
# Seção 4.1: Redimensiona imagens para acelerar o processamento
# EXECUTAR_REDIMENSIONAMENTO e MAX_DIMENSAO foram definidos na Seção 0.

from PIL import Image
from tqdm import tqdm

if not EXECUTAR_REDIMENSIONAMENTO:
    print("Redimensionamento desativado (EXECUTAR_REDIMENSIONAMENTO = False). Pulando.")
elif FONTE_DOS_DADOS != "UPLOAD_IMAGENS":
    print("Aviso: o redimensionamento só é aplicável para 'UPLOAD_IMAGENS'. Pulando.")
else:
    dest_dir = os.path.join(DATA_DIR, NOME_DA_CENA, "images")
    os.makedirs(dest_dir, exist_ok=True)

    image_files = [f for f in os.listdir(CAMINHO_ORIGEM) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Encontradas {len(image_files)} imagens para redimensionar (máx: {MAX_DIMENSAO}px)...")

    for filename in tqdm(image_files, desc="Redimensionando"):
        try:
            with Image.open(os.path.join(CAMINHO_ORIGEM, filename)) as img:
                width, height = img.size
                if width > MAX_DIMENSAO or height > MAX_DIMENSAO:
                    if width > height:
                        new_size = (MAX_DIMENSAO, int(height * MAX_DIMENSAO / width))
                    else:
                        new_size = (int(width * MAX_DIMENSAO / height), MAX_DIMENSAO)
                    img = img.resize(new_size, Image.Resampling.LANCZOS)
                img.save(os.path.join(dest_dir, filename))
        except Exception as e:
            print(f"Erro em {filename}: {e}")

    CAMINHO_ORIGEM = dest_dir
    print(f"\n✓ Redimensionamento concluído. Novo caminho de origem: {CAMINHO_ORIGEM}")

---
## Seção 5: Processamento de Dados com COLMAP

O COLMAP analisa as imagens/vídeo de `CAMINHO_ORIGEM` para estimar as **poses da câmera** e gerar uma **nuvem de pontos esparsa**. A saída é salva em `CAMINHO_DATASET` no formato que o Nerfstudio espera.

- **Seção 5.1** — Executa o processamento (pode demorar 15–60 min dependendo do dataset)
- **Seção 5.2** — Exibe o final do log para verificação de erros

In [ ]:
# Seção 5.1: Executa o processamento de dados com ns-process-data (COLMAP)

import os
import shutil
import subprocess
from distutils.dir_util import copy_tree

# Força o Qt a rodar sem interface gráfica (necessário em ambiente headless)
os.environ["QT_QPA_PLATFORM"] = "offscreen"

# Usa o SSD local para o processamento (muito mais rápido que o Drive)
# e copia os resultados para o Drive ao final
TMP_BASE = "/content/tmp_ns"
TMP_OUT = os.path.join(TMP_BASE, NOME_DA_CENA)

if os.path.exists(TMP_OUT):
    shutil.rmtree(TMP_OUT)
os.makedirs(TMP_OUT, exist_ok=True)

LOG_FILE = os.path.join(CAMINHO_DATASET, "processing.log")

print(f"Iniciando o processamento com COLMAP.")
print(f"  Origem    : {CAMINHO_ORIGEM}")
print(f"  Temporário: {TMP_OUT}  (SSD local)")
print(f"  Destino   : {CAMINHO_DATASET}  (Google Drive)")
print(f"  Log       : {LOG_FILE}")

# Seleciona o comando conforme tipo de entrada (diretório de imagens ou arquivo de vídeo)
if os.path.isdir(CAMINHO_ORIGEM):
    cmd_str = f"""
    xvfb-run -s "-screen 0 3840x2160x24" -a {NS_PREFIX} ns-process-data images \
        --data "{CAMINHO_ORIGEM}" \
        --output-dir "{TMP_OUT}" \
        --matching-method sequential \
        --no-gpu
    """
else:
    cmd_str = f"""
    xvfb-run -s "-screen 0 3840x2160x24" -a {NS_PREFIX} ns-process-data video \
        --data "{CAMINHO_ORIGEM}" \
        --output-dir "{TMP_OUT}" \
        --num-frames-to-sample 300 \
        --matching-method sequential \
        --no-gpu
    """

cmd_clean = " ".join(cmd_str.split())
print("\nExecutando comando...\n")

with open(LOG_FILE, "w") as f:
    try:
        subprocess.run(cmd_clean, shell=True, check=True, stdout=f, stderr=subprocess.STDOUT)
    except subprocess.CalledProcessError as e:
        print(f"\n✗ ERRO: o comando falhou (código {e.returncode}).")
        print(f"  Consulte o log para detalhes: {LOG_FILE}")
        raise e

print("✓ Processamento local concluído. Copiando para o Google Drive...")

try:
    copy_tree(TMP_OUT, CAMINHO_DATASET)
    print(f"✓ Dataset sincronizado em: {CAMINHO_DATASET}")
except Exception as e:
    print(f"✗ Erro ao copiar arquivos para o Drive: {e}")

In [ ]:
# Seção 5.2: Exibe o final do log de processamento para verificação rápida de erros
!tail -n 30 "{LOG_FILE}"

---
## Seção 6: Treinamento do Modelo

Configure o `localtunnel` na Seção 6.1 para acompanhar o progresso em tempo real no viewer web do Nerfstudio. Em seguida, execute **apenas uma** das células de treinamento:

| Célula | Modelo | Características |
|---|---|---|
| **6.2** | `nerfacto` | NeRF balanceado — melhor qualidade de imagem |
| **6.3** | `splatfacto` | Gaussian Splatting — treino rápido, ideal para exportação de `.ply` |

In [ ]:
# Seção 6.1: Inicia o viewer remoto via localtunnel
# Execute esta célula antes de iniciar o treinamento para obter o link de visualização.

import time

# Encerra qualquer túnel ativo anteriormente
!pkill -f localtunnel

# Abre um novo túnel na porta 7007 (porta padrão do Nerfstudio viewer)
get_ipython().system_raw('lt --port 7007 > url.txt 2>&1 &')
time.sleep(3)

try:
    with open('url.txt', 'r') as f:
        lt_url = f.read().strip().split('your url is: ')[-1]

    if lt_url:
        websocket_url = lt_url.replace("https://", "wss://")
        viewer_url = f"https://viewer.nerf.studio/?websocket_url={websocket_url}"
        print("✓ Viewer pronto! Acesse o link abaixo para acompanhar o treinamento em tempo real:")
        print(f"\n  {viewer_url}\n")
    else:
        print("✗ Falha ao criar o túnel. Log do localtunnel:")
        !cat url.txt
except FileNotFoundError:
    print("✗ Arquivo url.txt não encontrado. Verifique se o localtunnel foi instalado na Seção 1.2.")

✓ Viewer pronto! Acesse o link abaixo para acompanhar o treinamento em tempo real:

  https://viewer.nerf.studio/?websocket_url=wss://smart-vans-obey.loca.lt



In [ ]:

# Seção 6.2: Treinar com `nerfacto` (NeRF balanceado — melhor qualidade de imagem)
# O treinamento roda até 30.000 passos (padrão) ou até ser interrompido manualmente.

# Seção 6.2: Treinar com `nerfacto` (NeRF balanceado — melhor qualidade de imagem)

import subprocess
import re
import sys

def limpar_ansi(texto):
    """Remove sequências de escape ANSI do texto."""
    return re.sub(r'\x1b\[[0-9;]*[A-Za-z]', '', texto)

def is_linha_tabela(linha):
    """Detecta se a linha faz parte da tabela de progresso."""
    return any(x in linha for x in [
        "Step (% Done)", "---", "Train Iter", "Train Rays",
        "ETA (time)", "Viewer running"
    ])

cmd = (
    "conda run --no-capture-output -n ns ns-train nerfacto"
    f' --data "{CAMINHO_DATASET}"'
    f' --output-dir "{OUTPUTS_DIR}"'
    " --viewer.quit-on-train-completion True"
    " --viewer.make-share-url True"
    " --logging.local-writer.max-log-size 0"
    " --logging.steps-per-log 10"
)

buffer_tabela = []
dentro_tabela = False

with subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**__import__('os').environ, "PYTHONUNBUFFERED": "1"}
) as proc:
    for linha_raw in proc.stdout:
        linha = limpar_ansi(linha_raw).rstrip()

        if not linha:
            continue

        if is_linha_tabela(linha):
            # Acumula linhas da tabela no buffer
            if "Step (% Done)" in linha:
                buffer_tabela = [linha]   # reinicia a tabela
                dentro_tabela = True
            elif dentro_tabela:
                buffer_tabela.append(linha)
                # Exibe a tabela apenas quando estiver completa (linha separadora final)
                if linha.startswith("---") and len(buffer_tabela) > 3:
                    print("\n".join(buffer_tabela), flush=True)
                    buffer_tabela = []
                    dentro_tabela = False
        else:
            # Linhas fora da tabela (warnings, infos) são exibidas diretamente
            dentro_tabela = False
            buffer_tabela = []
            print(linha, flush=True)

if proc.returncode not in (0, None):
    raise RuntimeError(f"✗ ns-train falhou com código {proc.returncode}")


In [17]:

# Seção 6.3: Treinar com `splatfacto` (Gaussian Splatting — treino rápido, exportação .ply)
# --pipeline.datamanager.cache-images disk reduz o uso de RAM ao processar frames do vídeo.


import subprocess
import re
import sys

def limpar_ansi(texto):
    """Remove sequências de escape ANSI do texto."""
    return re.sub(r'\x1b\[[0-9;]*[A-Za-z]', '', texto)

def is_linha_tabela(linha):
    """Detecta se a linha faz parte da tabela de progresso."""
    return any(x in linha for x in [
        "Step (% Done)", "---", "Train Iter", "Train Rays",
        "ETA (time)", "Viewer running"
    ])

cmd = (
    "conda run --no-capture-output -n ns ns-train splatfacto"
    f' --data "{CAMINHO_DATASET}"'
    f' --output-dir "{OUTPUTS_DIR}"'
    " --viewer.quit-on-train-completion True"
    " --viewer.make-share-url True"
    " --logging.local-writer.max-log-size 0"
    " --logging.steps-per-log 10"
)

buffer_tabela = []
dentro_tabela = False

with subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**__import__('os').environ, "PYTHONUNBUFFERED": "1"}
) as proc:
    for linha_raw in proc.stdout:
        linha = limpar_ansi(linha_raw).rstrip()

        if not linha:
            continue

        if is_linha_tabela(linha):
            # Acumula linhas da tabela no buffer
            if "Step (% Done)" in linha:
                buffer_tabela = [linha]   # reinicia a tabela
                dentro_tabela = True
            elif dentro_tabela:
                buffer_tabela.append(linha)
                # Exibe a tabela apenas quando estiver completa (linha separadora final)
                if linha.startswith("---") and len(buffer_tabela) > 3:
                    print("\n".join(buffer_tabela), flush=True)
                    buffer_tabela = []
                    dentro_tabela = False
        else:
            # Linhas fora da tabela (warnings, infos) são exibidas diretamente
            dentro_tabela = False
            buffer_tabela = []
            print(linha, flush=True)

if proc.returncode not in (0, None):
    raise RuntimeError(f"✗ ns-train falhou com código {proc.returncode}")

/usr/local/envs/ns/lib/python3.10/site-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/envs/ns/lib/python3.10/site-packages/nerfstudio/field_components/activations.py:39: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, g):
[18:00:40] Using --data alias for --data.pipeline.datamanager.data                                          train.py:230
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('/content/drive/MyDrive/Faculdade/TCC/desenvolvimento/outputs'),
    method_name='splatfacto',
    experiment_name=None,
    pro

KeyboardInterrupt: 

---
## Seção 7: Exportação e Geração de Artefatos

Após o treinamento, use o modelo para gerar diferentes saídas:

| Célula | Ação |
|---|---|
| **7.1** | Localiza automaticamente o `config.yml` do último treinamento |
| **7.2** | Exporta a nuvem de pontos Gaussian Splat (`.ply`) — apenas para `splatfacto` |
| **7.3** | Renderiza um vídeo de trajetória de câmera em espiral |
| **7.4** | Compacta todos os artefatos exportados em um `.zip` para download |

In [ ]:
# Seção 7.1: Localiza o config.yml do treinamento mais recente em OUTPUTS_DIR

import glob
import os

configs_encontrados = sorted(
    glob.glob(os.path.join(OUTPUTS_DIR, "**", "config.yml"), recursive=True),
    key=os.path.getmtime
)

if not configs_encontrados:
    raise FileNotFoundError(f"Nenhum 'config.yml' encontrado em: {OUTPUTS_DIR}")

CAMINHO_CONFIG_TREINO = configs_encontrados[-1]
print(f"✓ Configuração do último treinamento:\n  {CAMINHO_CONFIG_TREINO}")

In [ ]:
# Seção 7.2: Exporta a nuvem de pontos Gaussian Splat (.ply)
# Só funciona se o modelo treinado for 'splatfacto'.

import yaml

with open(CAMINHO_CONFIG_TREINO, 'r') as f:
    config_data = yaml.safe_load(f)

if config_data.get("_target", "").endswith("splatfacto"):
    CAMINHO_EXPORT_GS = os.path.join(EXPORTS_DIR, NOME_DA_CENA, "gaussian_splat")
    os.makedirs(CAMINHO_EXPORT_GS, exist_ok=True)
    print("Modelo 'splatfacto' detectado. Exportando Gaussian Splat...")

    !{NS_PREFIX} ns-export gaussian-splat \
        --load-config "{CAMINHO_CONFIG_TREINO}" \
        --output-dir "{CAMINHO_EXPORT_GS}"

    print(f"✓ Arquivo .ply salvo em: {CAMINHO_EXPORT_GS}")
else:
    print("Aviso: o modelo treinado não é 'splatfacto'. Pulando a exportação de Gaussian Splat.")

In [ ]:
# Seção 7.3: Renderiza um vídeo com trajetória de câmera em espiral

CAMINHO_EXPORT_VIDEO = os.path.join(EXPORTS_DIR, NOME_DA_CENA, "renders")
os.makedirs(CAMINHO_EXPORT_VIDEO, exist_ok=True)
VIDEO_PATH = os.path.join(CAMINHO_EXPORT_VIDEO, "render_spiral.mp4")

print("Iniciando renderização do vídeo. Isso pode demorar alguns minutos...")

!{NS_PREFIX} ns-render camera-path \
    --load-config "{CAMINHO_CONFIG_TREINO}" \
    --camera-path-filename "{CAMINHO_DATASET}/camera_path.json" \
    --output-path "{VIDEO_PATH}"

print(f"✓ Vídeo salvo em: {VIDEO_PATH}")

In [ ]:
# Seção 7.4: Compacta todos os artefatos exportados em um .zip para download

import zipfile

CAMINHO_BASE_EXPORT = os.path.join(EXPORTS_DIR, NOME_DA_CENA)
CAMINHO_ZIP = f"{CAMINHO_BASE_EXPORT}_exports.zip"

print(f"Criando arquivo zip em:\n  {CAMINHO_ZIP}")

with zipfile.ZipFile(CAMINHO_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(CAMINHO_BASE_EXPORT):
        for filename in files:
            file_path = os.path.join(root, filename)
            zf.write(file_path, os.path.relpath(file_path, CAMINHO_BASE_EXPORT))

print(f"✓ Compactação concluída: {CAMINHO_ZIP}")